# GC-LSTM-style — baseline Final64
Notebook ini adalah implementasi aktif. Loss menggunakan jumlah squared error valid dibagi jumlah target valid, konsisten dengan GWN.
Perbaikan kode belum sama dengan hasil retraining. Run baru disimpan di `outputs/final_64/gclstm_baseline_corrected`; hasil historis `gclstm_baseline` tetap menjadi sumber tabel lama sampai analisis diperbarui.
Tahap training MAE awal dan pengulangan training seed 42 telah dihapus. Jalankan pipeline lima seed di bawah satu kali.


# GC-LSTM Baseline on Final64

Notebook ini digunakan untuk menjalankan baseline GC-LSTM pada dataset Final64.

Tujuan:
- menggunakan input/output yang sama dengan eksperimen Graph WaveNet;
- input: 12 jam historical `current_speed`;
- output: prediksi `t+1`, `t+3`, dan `t+6`;
- menggunakan fixed geographic adjacency sebagai graph awal;
- mengevaluasi MAE, RMSE, parameter count, dan training time;
- evaluasi menggunakan seed 42, 123, 456, 789, dan 2026.

Dataset:
- 64 corridor
- Train: 415 window
- Validation: 92 window
- Test: 93 window

In [ ]:
from pathlib import Path
import os
import time
import random

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError) as error:
    print(f"Deterministic operations unavailable: {error}")

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

## Load Final64 Dataset dan Fixed Graph

Cell ini memuat:
- data train, validation, dan test Final64;
- scaler;
- fixed geographic adjacency yang sama dengan eksperimen Graph WaveNet.

In [ ]:
# Path setup otomatis: cari project root

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path):
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (path / "data" / "final_64").exists() and (path / "outputs" / "final_64").exists():
            return path

    raise FileNotFoundError("Project root tidak ditemukan dari current working directory.")


ROOT = find_project_root(CURRENT_DIR)

DATA_DIR = ROOT / "data" / "final_64"
GRAPH_PATH = ROOT / "outputs" / "final_64" / "fixed_graph" / "fixed_graph_64.npz"

TRAIN_PATH = DATA_DIR / "stgnn_train.npz"
VAL_PATH = DATA_DIR / "stgnn_val.npz"
TEST_PATH = DATA_DIR / "stgnn_test.npz"
SCALER_PATH = DATA_DIR / "stgnn_scaler.npz"

print("Current dir :", CURRENT_DIR)
print("Project root:", ROOT)
print()

print("Train exists :", TRAIN_PATH.exists())
print("Val exists   :", VAL_PATH.exists())
print("Test exists  :", TEST_PATH.exists())
print("Scaler exists:", SCALER_PATH.exists())
print("Graph exists :", GRAPH_PATH.exists())

## Load Dataset Arrays

Memuat array train, validation, test, scaler, dan fixed adjacency.

In [ ]:
# Load Final64
train_data = np.load(TRAIN_PATH, allow_pickle=True)
val_data = np.load(VAL_PATH, allow_pickle=True)
test_data = np.load(TEST_PATH, allow_pickle=True)

scaler_data = np.load(SCALER_PATH, allow_pickle=True)
graph_data = np.load(GRAPH_PATH, allow_pickle=True)

print("Train keys :", train_data.files)
print("Val keys   :", val_data.files)
print("Test keys  :", test_data.files)
print("Scaler keys:", scaler_data.files)
print("Graph keys :", graph_data.files)

print("\nShapes:")
print("X_train:", train_data["X"].shape)
print("y_train:", train_data["y"].shape)
print("mask_train:", train_data["mask"].shape)

print("X_val  :", val_data["X"].shape)
print("y_val  :", val_data["y"].shape)

print("X_test :", test_data["X"].shape)
print("y_test :", test_data["y"].shape)

print("\nGraph:")
print("adjacency:", graph_data["adjacency"].shape)
print("directed_knn_adjacency:", graph_data["directed_knn_adjacency"].shape)

print("\nScaler:")
print("mean :", scaler_data["mean"])
print("scale:", scaler_data["scale"])

## Prepare Fixed Adjacency dan Dataset

GC-LSTM baseline menggunakan fixed geographic graph yang sama dengan eksperimen Graph WaveNet.

Adjacency dinormalisasi agar propagasi informasi antarnode lebih stabil.

In [ ]:
# Prepare dataset arrays
X_train = train_data["X"].astype(np.float32)
y_train = train_data["y"].astype(np.float32)
mask_train = train_data["mask"].astype(np.float32)

X_val = val_data["X"].astype(np.float32)
y_val = val_data["y"].astype(np.float32)
mask_val = val_data["mask"].astype(np.float32)

X_test = test_data["X"].astype(np.float32)
y_test = test_data["y"].astype(np.float32)
mask_test = test_data["mask"].astype(np.float32)

# Fixed geographic adjacency
A = graph_data["directed_knn_adjacency"].astype(np.float32)

# Tambahkan self-loop
I = np.eye(A.shape[0], dtype=np.float32)
A_hat = A + I

# Row normalization
row_sum = A_hat.sum(axis=1, keepdims=True)
A_norm = A_hat / np.maximum(row_sum, 1e-8)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("mask_train:", mask_train.shape)

print("\nAdjacency:")
print("A shape:", A.shape)
print("A_norm shape:", A_norm.shape)
print("A_norm row sum min:", A_norm.sum(axis=1).min())
print("A_norm row sum max:", A_norm.sum(axis=1).max())

print("\nNaN check:")
print("X_train NaN:", np.isnan(X_train).sum())
print("y_train NaN:", np.isnan(y_train).sum())

## Prepare Masked Targets

Walaupun target saat ini tidak mengandung NaN, mask tetap digunakan agar hanya target valid yang berkontribusi pada loss dan evaluasi.

In [ ]:
# Pastikan mask dalam format float32
mask_train = mask_train.astype(np.float32)
mask_val = mask_val.astype(np.float32)
mask_test = mask_test.astype(np.float32)

# Cek jumlah target valid per split
print("Valid targets:")
print("Train:", int(mask_train.sum()))
print("Val  :", int(mask_val.sum()))
print("Test :", int(mask_test.sum()))

print("\nValid target per horizon (test):")
for h, label in enumerate(["t+1", "t+3", "t+6"]):
    print(label, ":", int(mask_test[:, h, :].sum()))

# Sanity check
print("\nMask unique values:")
print("Train:", np.unique(mask_train))
print("Val  :", np.unique(mask_val))
print("Test :", np.unique(mask_test))

## GC-LSTM Baseline Architecture

Alur model:

1. Setiap timestep memiliki fitur `current_speed` untuk 64 corridor.
2. Fitur dipropagasikan menggunakan fixed normalized adjacency.
3. Hasil graph propagation dari 12 timestep disusun sebagai sequence.
4. LSTM mempelajari pola temporal dari sequence tersebut.
5. Dense output menghasilkan prediksi untuk horizon t+1, t+3, dan t+6 pada 64 corridor.

In [ ]:
from tensorflow.keras import layers, Model

NUM_NODES = 64
INPUT_STEPS = 12
NUM_HORIZONS = 3

GC_UNITS = 32
LSTM_UNITS = 64
DROPOUT = 0.3


class GraphPropagation(layers.Layer):
    def __init__(self, adjacency, units, **kwargs):
        super().__init__(**kwargs)

        self.adjacency = tf.constant(adjacency, dtype=tf.float32)

        self.proj = layers.Dense(units, activation="relu")

    def call(self, x):
        # x shape:
        # (batch, time, nodes, features)

        x = tf.einsum("ij,btjf->btif", self.adjacency, x)

        x = self.proj(x)

        return x


def build_gclstm():
    inputs = layers.Input(shape=(INPUT_STEPS, NUM_NODES, 1), name="traffic_input")

    # Graph propagation
    x = GraphPropagation(A_norm, GC_UNITS, name="graph_propagation")(inputs)

    # Flatten node × graph feature per timestep
    x = layers.Reshape((INPUT_STEPS, NUM_NODES * GC_UNITS), name="flatten_graph_features")(x)

    # Temporal modeling
    x = layers.LSTM(LSTM_UNITS, return_sequences=False, name="temporal_lstm")(x)

    x = layers.Dropout(DROPOUT, name="dropout")(x)

    # Multi-horizon output
    x = layers.Dense(NUM_HORIZONS * NUM_NODES, name="forecast_dense")(x)

    outputs = layers.Reshape((NUM_HORIZONS, NUM_NODES), name="forecast_output")(x)

    model = Model(inputs=inputs, outputs=outputs, name="GC_LSTM_Final64")

    return model


gclstm = build_gclstm()

gclstm.summary()

print("\nTotal parameters:")
print(gclstm.count_params())

In [ ]:
OUTPUT_DIR = ROOT / "outputs" / "final_64" / "gclstm_baseline_corrected"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# INVERSE SCALING HELPER
# ============================================================

SCALER_PATH = ROOT / "data" / "final_64" / "stgnn_scaler.npz"

scaler_data = np.load(SCALER_PATH, allow_pickle=True)

SCALER_MEAN = float(scaler_data["mean"][0])

SCALER_SCALE = float(scaler_data["scale"][0])


def inverse_scale(x):
    return x * SCALER_SCALE + SCALER_MEAN


print("Scaler mean :", SCALER_MEAN)
print("Scaler scale:", SCALER_SCALE)

## Multi-Seed GC-LSTM-style Evaluation

GC-LSTM-style dievaluasi menggunakan lima random seed yang sama dengan eksperimen Graph WaveNet:

- 42
- 123
- 456
- 789
- 2026

Tujuannya adalah mengevaluasi stabilitas performa model dan membuat perbandingan yang lebih adil terhadap Fixed, Adaptive, dan Hybrid Graph WaveNet.

Konfigurasi training dipertahankan sama untuk setiap seed.

In [ ]:
# ============================================================
# GC-LSTM-style MULTI-SEED
# ============================================================

import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEEDS = [42, 123, 456, 789, 2026]

multi_seed_rows = []
multi_seed_timing = []


# ============================================================
# ELEMENTWISE SQUARED ERROR
# Valid-target normalization matches Graph WaveNet; targets carry their masks.
# ============================================================


def masked_mse(y_true, y_pred):
    values, mask = y_true[..., 0], y_true[..., 1]
    return tf.reduce_sum(tf.square(values - y_pred) * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)


# ============================================================
# RUN EACH SEED
# ============================================================

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"GC-LSTM-style | SEED {seed}")
    print("=" * 60)

    # --------------------------------------------------------
    # RESET
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except (AttributeError, RuntimeError) as error:
        print(f"Deterministic operations unavailable: {error}")

    # --------------------------------------------------------
    # BUILD MODEL
    # --------------------------------------------------------

    model = build_gclstm()

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=masked_mse)

    # --------------------------------------------------------
    # CALLBACKS
    # --------------------------------------------------------

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True
    )

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    start_time = time.perf_counter()

    history = model.fit(
        X_train,
        np.stack([y_train, mask_train], axis=-1),
        validation_data=(X_val, np.stack([y_val, mask_val], axis=-1)),
        epochs=300,
        batch_size=32,
        callbacks=[early_stop, reduce_lr],
        verbose=0,
    )

    total_time = time.perf_counter() - start_time

    epochs_run = len(history.history["loss"])

    best_epoch = int(np.argmin(history.history["val_loss"])) + 1

    best_val_loss = float(np.min(history.history["val_loss"]))

    mean_sec_epoch = total_time / epochs_run

    print("epochs:", epochs_run)

    print("best epoch:", best_epoch)

    print("best val loss:", round(best_val_loss, 6))

    # --------------------------------------------------------
    # TEST PREDICTION
    # --------------------------------------------------------

    model.save_weights(OUTPUT_DIR / f"gclstm_seed{seed}.weights.h5")
    pd.DataFrame(history.history).to_csv(OUTPUT_DIR / f"gclstm_seed{seed}_history.csv", index=False)

    pred_std = model(X_test, training=False).numpy()

    pred_real = inverse_scale(pred_std)

    actual_real = inverse_scale(y_test)

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    for h_idx, horizon in enumerate(["t+1", "t+3", "t+6"]):
        valid = mask_test[:, h_idx, :] == 1

        actual = actual_real[:, h_idx, :][valid]

        pred = pred_real[:, h_idx, :][valid]

        mae = np.mean(np.abs(actual - pred))

        rmse = np.sqrt(np.mean(np.square(actual - pred)))

        multi_seed_rows.append(
            {
                "model": "GC-LSTM-style",
                "seed": seed,
                "horizon": horizon,
                "mae": float(mae),
                "rmse": float(rmse),
                "valid_targets": int(valid.sum()),
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "parameters": model.count_params(),
            }
        )

        print(horizon, "| MAE:", round(mae, 6), "| RMSE:", round(rmse, 6))

    # --------------------------------------------------------
    # TIMING
    # --------------------------------------------------------

    multi_seed_timing.append(
        {
            "model": "GC-LSTM-style",
            "seed": seed,
            "epochs_run": epochs_run,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "mean_sec_per_epoch": mean_sec_epoch,
            "total_training_time_sec": total_time,
            "parameters": model.count_params(),
        }
    )


# ============================================================
# DATAFRAMES
# ============================================================

gclstm_multiseed = pd.DataFrame(multi_seed_rows)

gclstm_multiseed_timing = pd.DataFrame(multi_seed_timing)

print("\n=== ALL GC-LSTM MULTI-SEED RESULTS ===")

display(gclstm_multiseed.round(6))

print("\n=== TRAINING TIMING ===")

display(gclstm_multiseed_timing.round(4))

In [ ]:
# ============================================================
# GC-LSTM MULTI-SEED SUMMARY
# ============================================================

gclstm_multiseed_summary = (
    gclstm_multiseed.groupby("horizon")
    .agg(
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
    )
    .reset_index()
)

print("=== GC-LSTM-style 5-SEED SUMMARY ===")

display(gclstm_multiseed_summary.round(6))

In [ ]:
MULTISEED_RESULT_PATH = OUTPUT_DIR / "gclstm_fair_multiseed_results.csv"

MULTISEED_SUMMARY_PATH = OUTPUT_DIR / "gclstm_fair_multiseed_summary.csv"

MULTISEED_TIMING_PATH = OUTPUT_DIR / "gclstm_fair_multiseed_timing.csv"

gclstm_multiseed.to_csv(MULTISEED_RESULT_PATH, index=False)

gclstm_multiseed_summary.to_csv(MULTISEED_SUMMARY_PATH, index=False)

gclstm_multiseed_timing.to_csv(MULTISEED_TIMING_PATH, index=False)

print("Saved:")
print(MULTISEED_RESULT_PATH)
print(MULTISEED_SUMMARY_PATH)
print(MULTISEED_TIMING_PATH)